# Building custom HypEx experiments: an architecture textbook

Ready-made shells are only the outermost layer. This chapter explains the state,
execution and reporting contracts well enough to assemble a reproducible analysis
and add a new statistical block without modifying the library.

## Learning objectives

- explain what lives in each `ExperimentData` namespace;
- predict which blocks replace data and which only append results;
- choose a sensible preprocessing order;
- implement a `GroupsComparator` and retrieve its output by executor id;
- compose per-target and per-segment experiments;
- expose custom and standard results through one `Output` object.

**Prerequisites:** basic A/B concepts and Python classes. The statistical example
uses a median difference because its implementation is small but its estimand is
different enough to require careful interpretation.

## The mental model: a typed state machine

An executor receives state and returns state. It does not need to know the
whole analysis, only its inputs and where its outputs belong. An
`Experiment` supplies order; roles supply column semantics; executor ids
supply addressable results; reporters turn those results into a public API.

This gives three useful extension points:

1. **new data semantics** through roles and transformations;
2. **new computations** through executors;
3. **new presentation** through reporters and outputs.

Most custom analyses should compose these pieces rather than fork a shell.

## 1. The architecture in one picture

```text
Dataset + roles
      │
      ▼
ExperimentData ── ds / additional_fields / analysis_tables / variables / groups
      │
      ▼
Experiment ── ordered executors
      │
      ▼
Reporter ── reads results by executor id
      │
      ▼
ExperimentShell.execute(...) ── Output.summary + additional reports
```

Only transformers replace the working `ds`. Encoders write derived columns
to `additional_fields`; comparators and analyzers write result tables.
This separation is what makes blocks composable without hiding state.

### The five namespaces are not interchangeable

| Namespace | Typical writer | Lifetime / purpose |
|---|---|---|
| `ds` | transformers | Working table used by downstream blocks |
| `additional_fields` | splitters, encoders, ML executors | Derived columns aligned with the original index |
| `analysis_tables` | comparators, analyzers, experiment containers | Small statistical or reporting tables |
| `variables` | operators and conditional executors | Scalar or nested results that are not tables |
| `groups` | splitters and distance calculations | Cached group slices or transformed feature spaces |

The index is part of the contract: matching indexes and additional columns
are meaningful only if row alignment is preserved. Resetting an index for
cosmetic reasons can silently join the wrong counterpart.

`SmallDataset` result tables remain pandas-based even on Spark. Distributed
raw data and small driver-side reports are deliberately different objects.

In [1]:
from typing import Any

import numpy as np
import pandas as pd

from hypex.analyzers.ab import ABAnalyzer
from hypex.comparators import (
    Chi2Test,
    GroupDifference,
    GroupsComparator,
    GroupSizes,
    TTest,
    UTest,
)
from hypex.dataset import (
    Dataset,
    ExperimentData,
    FeatureRole,
    GroupingRole,
    InfoRole,
    TargetRole,
    TreatmentRole,
)
from hypex.experiments import Experiment, GroupExperiment, OnRoleExperiment
from hypex.reporters import ABTestReporter, Reporter
from hypex.transformers import (
    CategoryAggregator,
    ConstFilter,
    CVFilter,
    DummyEncoder,
    Float32Caster,
    NaDropper,
    NanFilter,
)
from hypex.ui.base import ExperimentShell, Output
from hypex.utils import ID_SPLIT_SYMBOL, ABNTestMethodsEnum, ExperimentDataEnum

rng = np.random.default_rng(123)
n_users = 1_500
age = rng.normal(38, 11, n_users).round()
sessions = rng.poisson(7, n_users).astype(float)
city = rng.choice(
    ["Moscow", "Kazan", "Other", "Tinyville"],
    n_users,
    p=[0.55, 0.30, 0.14, 0.01],
)
treat = rng.binomial(1, 0.5, n_users)
revenue = 25 + 2 * age + 4 * sessions + 10 * treat + rng.normal(0, 18, n_users)
converted = rng.binomial(1, np.clip(0.03 + revenue / 500, 0, 0.8))
channel = np.where(
    treat == 1,
    rng.choice(["organic", "paid"], n_users, p=[0.65, 0.35]),
    rng.choice(["organic", "paid"], n_users, p=[0.50, 0.50]),
)
segment = rng.choice(["retail", "saas"], n_users, p=[0.65, 0.35])
custom_df = pd.DataFrame(
    {
        "user_id": np.arange(n_users),
        "treat": treat,
        "revenue": revenue,
        "converted": converted,
        "channel": channel,
        "segment": segment,
        "age": age,
        "age_copy": age + rng.normal(0, 0.01, n_users),
        "sessions": sessions,
        "city": city,
        "almost_constant": np.where(np.arange(n_users) < 20, 1, 0),
    }
)
custom_df.loc[custom_df.sample(frac=0.04, random_state=9).index, "age"] = np.nan
custom_df.head()

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


,user_id,treat,revenue,converted,channel,segment,age,age_copy,sessions,city,almost_constant
0,0,1,83.214769,0,paid,retail,27.0,26.992673,5.0,Moscow,1
1,1,0,133.037663,0,paid,retail,34.0,33.999613,6.0,Moscow,1
2,2,0,143.168004,1,paid,retail,52.0,52.006064,8.0,Moscow,1
3,3,1,155.885627,0,organic,retail,40.0,40.004774,6.0,Kazan,1
4,4,1,140.236037,0,organic,saas,48.0,47.990498,3.0,Moscow,1


In [2]:
custom_data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "revenue": TargetRole(float),
        "converted": TargetRole(int),
        "channel": TargetRole(str),
        "segment": GroupingRole(str),
        "age": FeatureRole(float),
        "age_copy": FeatureRole(float),
        "sessions": FeatureRole(float),
        "city": FeatureRole(str),
        "almost_constant": FeatureRole(int),
    },
    data=custom_df,
)

## 2. Preprocessing is a pipeline with semantics

The chain below drops rows with missing features, collapses rare categories,
disables low-variation and almost-constant features, downcasts numeric
features, and one-hot encodes the remaining categoricals.

Filters generally change roles to `InfoRole` rather than silently deleting
source columns. `DummyEncoder` writes new columns to `additional_fields`.
The enclosing `Experiment` detects transformers and deep-copies its input,
so the original dataset is not mutated.

> In production, decide row filtering and outlier handling from the causal
> question. Dropping customers based on a post-treatment outcome can change
> the estimand or introduce selection bias.

### Why this preprocessing order is deliberate

- **Missing rows first.** `NaDropper` prevents later statistics and distance
  calculations from receiving undefined values.
- **Rare categories next.** `CategoryAggregator` limits sparse chi-square
  cells and one-hot expansion.
- **Role filters next.** `NanFilter`, `ConstFilter` and `CVFilter` disable
  columns that cannot contribute stable information.
- **Numeric casting before modelling.** `Float32Caster` reduces memory but
  is a precision trade-off, not a statistical transformation.
- **Encoding last.** `DummyEncoder` creates numeric additional fields while
  preserving the original categoricals for chi-square tests.

Missingness itself needs a causal reading. MCAR means missingness is
independent of everything; MAR means it is explained by observed variables;
MNAR means it still depends on unobserved values. Dropping rows is easiest
to defend under MCAR. Under MAR or MNAR, it can change the target
population and create selection bias.

For categorical level $k$, dummy encoding uses $k-1$ indicators and leaves
one reference level. Keeping all $k$ indicators plus an intercept creates
exact linear dependence—the dummy-variable trap.

In [3]:
preprocessing = Experiment(
    executors=[
        NaDropper(target_roles=FeatureRole(), how="any"),
        CategoryAggregator(
            target_roles=FeatureRole(), threshold=30, new_group_name="Other"
        ),
        NanFilter(target_roles=FeatureRole(), threshold=0.5),
        ConstFilter(target_roles=FeatureRole(), threshold=0.98),
        CVFilter(target_roles=FeatureRole(), lower_bound=0.01),
        Float32Caster(target_roles=[FeatureRole()]),
        DummyEncoder(target_roles=FeatureRole()),
    ]
)
prepared = preprocessing.execute(ExperimentData(custom_data))
prepared.ds.roles

{'user_id': Info(<class 'int'>),
 'treat': Treatment(<class 'int'>),
 'revenue': Target(<class 'float'>),
 'converted': Target(<class 'int'>),
 'channel': Target(<class 'str'>),
 'segment': Grouping(<class 'str'>),
 'age': Feature(data_type=<class 'float'>),
 'age_copy': Feature(data_type=<class 'float'>),
 'sessions': Feature(data_type=<class 'float'>),
 'city': Feature(data_type=<class 'str'>),
 'almost_constant': Info(<class 'int'>),
 'DummyEncoder┴┴_city_Moscow': AdditionalFeature(data_type=<class 'bool'>),
 'DummyEncoder┴┴_city_Other': AdditionalFeature(data_type=<class 'bool'>)}

In [4]:
print("Original feature columns:", custom_data.search_columns(FeatureRole()))
print("Prepared feature columns:", prepared.ds.search_columns(FeatureRole()))
print("Additional columns:", prepared.additional_fields.columns)
prepared.additional_fields.raw_data.head()

Original feature columns: ['age', 'age_copy', 'sessions', 'city', 'almost_constant']
Prepared feature columns: ['age', 'age_copy', 'sessions', 'city', 'DummyEncoder┴┴_city_Moscow', 'DummyEncoder┴┴_city_Other']
Additional columns: ['DummyEncoder┴┴_city_Moscow', 'DummyEncoder┴┴_city_Other']


,DummyEncoder┴┴_city_Moscow,DummyEncoder┴┴_city_Other
0,True,False
1,True,False
2,True,False
3,False,False
4,True,False


### Inspect roles, not only values

`almost_constant` now has `InfoRole`: it is still visible in the table but
no longer selected as a feature. Dummy columns have
`AdditionalFeatureRole` and appear in `additional_fields`.

This distinction matters when a later result is “missing”. First ask whether
a filter changed the role, then whether an encoder wrote to another
namespace. Searching only physical column names hides both explanations.

## 3. Add a robust comparator without modifying HypEx

Means are not the whole story for heavy-tailed revenue. A median difference
is a useful descriptive complement, although it answers a different question
and does not inherit the t-test's null hypothesis.

`GroupsComparator` already handles role resolution, group splitting and
result storage. We only restrict eligible dtypes and implement the metric.
The first sorted treatment value remains the baseline.

### Mean and median differences estimate different quantities

The mean difference is
$\mathbb E[Y_t]-\mathbb E[Y_c]$. The median difference is
$Q_{0.5}(Y_t)-Q_{0.5}(Y_c)$, where $Q_p$ is a quantile. In general,

$$Q_{0.5}(Y_t)-Q_{0.5}(Y_c)
\neq Q_{0.5}(Y_t-Y_c).$$

Medians reduce the influence of extreme observations, but a median
difference is not automatically a replacement for the business mean,
especially for revenue whose total depends on $\mathbb E[Y]$.

The custom block therefore returns descriptive statistics, not a p-value.
To test a median hypothesis we would need a separate inferential procedure,
such as an appropriate bootstrap or quantile model.

In [5]:
class GroupMedianDifference(GroupsComparator):
    @property
    def search_types(self):
        return [int, float]

    @classmethod
    def _inner_function(
        cls,
        data: Dataset,
        test_data: Dataset | None = None,
        **kwargs: Any,
    ) -> dict[str, float]:
        test_data = cls._check_test_data(test_data)
        control_median = float(data.quantile(0.5))
        test_median = float(test_data.quantile(0.5))
        return {
            "control median": control_median,
            "test median": test_median,
            "median difference": test_median - control_median,
        }

### Walk through the custom class

- `search_types` limits the block to numeric targets.
- `_check_test_data` enforces the two-sample contract.
- `data.quantile(0.5)` computes the median through the backend-agnostic
  `Dataset` API.
- The returned dictionary becomes columns in an analysis table.
- The base class resolves `TreatmentRole`, sorts groups and stores the
  result under `self.id`.

The id is not decorative. Its logical shape is

```text
ClassName ┴ parameters_hash ┴ key
```

`OnRoleExperiment` places the target name in `key`, preventing revenue and
conversion results from overwriting one another.

In [6]:
# A standalone instance shows the id structure before any pipeline runs.
example_median_executor = GroupMedianDifference(
    compare_by="groups", grouping_role=TreatmentRole()
)
print("Executor id:", example_median_executor.id)
print("Display-safe id:", example_median_executor.id_for_name)

Executor id: GroupMedianDifference┴┴
Display-safe id: GroupMedianDifference__


## 4. Compose the analytical pipeline

`OnRoleExperiment` runs the comparators for each target. On pandas, raw-data
tests iterate per column; aggregate-first comparators can process several
targets in one pass. `ABAnalyzer` then collects test results and performs
the configured multiple-testing correction.

The reporter below finds custom tables through `ExperimentData.get_ids`.
Executor ids contain the target name, which lets us preserve it in the
final table without hard-coding a physical column in the executor itself.

### Why reporters search by class and id

Hard-coding a result key couples a reporter to one physical run. Searching
`ExperimentData` by `GroupMedianDifference` instead preserves composition:
a reporter can find every table produced by that executor family and let
the id supply the target name.

Pipeline order is semantic:

```text
GroupSizes
    → per-target GroupDifference / GroupMedianDifference / tests
    → ABAnalyzer
    → reporters
```

Moving `ABAnalyzer` before the tests would leave it with no results. The
framework does not turn every ordering mistake into an exception; an empty
upstream id list can produce an empty report instead.

In [7]:
class MedianReporter(Reporter):
    def report(self, data: ExperimentData):
        ids = data.get_ids(
            GroupMedianDifference,
            searched_space=ExperimentDataEnum.analysis_tables,
        )["GroupMedianDifference"]["analysis_tables"]
        tables = []
        for executor_id in ids:
            feature = executor_id.split(ID_SPLIT_SYMBOL)[-1]
            table = data.analysis_tables[executor_id].add_column(
                feature, role={"feature": InfoRole()}
            )
            tables.append(table)
        return tables[0].append(tables[1:]) if len(tables) > 1 else tables[0]


analysis = Experiment(
    executors=[
        GroupSizes(grouping_role=TreatmentRole()),
        OnRoleExperiment(
            executors=[
                GroupDifference(grouping_role=TreatmentRole()),
                GroupMedianDifference(
                    compare_by="groups", grouping_role=TreatmentRole()
                ),
                TTest(grouping_role=TreatmentRole()),
                UTest(grouping_role=TreatmentRole()),
                Chi2Test(grouping_role=TreatmentRole()),
            ],
            role=TargetRole(),
        ),
        ABAnalyzer(multitest_method=ABNTestMethodsEnum.holm),
    ]
)

In [8]:
shell = ExperimentShell(
    experiment=analysis,
    output=Output(
        summary_reporter=ABTestReporter(invert_pass=True),
        additional_reporters={"median": MedianReporter()},
    ),
)
custom_result = shell.execute(prepared)
custom_result.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,UTest pass,UTest p-value,Chi2Test pass,Chi2Test p-value
0,revenue,1,129.118131,139.004016,9.885884,7.656465,6.739324,13.032445,OK,9.549148e-10,OK,7.372462e-09,NaN,NaN
1,converted,1,0.269394,0.310534,0.041140,15.271342,-0.005707,0.087987,NOT OK,8.542528e-02,NOT OK,8.563838e-02,NaN,NaN
2,channel,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,0.004009


In [9]:
custom_result.median

,control median,test median,median difference,feature
1,127.922919,138.444202,10.521284,revenue
1,0.000000,0.000000,0.000000,converted


### Compare the outputs without conflating them

The standard summary answers mean and distribution questions. The median
table answers a robust-location question. A large mean difference beside a
small median difference can indicate changed tails or a few extreme values;
it does not prove that one estimator is “wrong”.

For converted, both medians can be zero while the conversion-rate means
differ. Binary quantiles are coarse, which is another reason to choose an
estimand that matches the metric.

`ExperimentShell` deep-copies a raw `Dataset` before execution and manages
Spark persistence. The `Output` object is reused by the shell, so a second
`execute` call replaces the previous result attributes.

Notice the two levels of output: the standard reporter knows how to format
built-in tests, while the additional reporter exposes our custom metric.
For a reusable package, this same pattern can become a named
`ExperimentShell` subclass with constructor arguments forwarded through
`experiment.set_params`.

An `Output` instance belongs to its shell and is overwritten on the next
`execute` call. Create separate shells when you need to retain several
results side by side.

## 5. Repeat the whole pipeline per segment

`GroupExperiment` splits on `GroupingRole`, runs a fresh pipeline in every
segment, and prefixes report columns with the segment name. This is not a
substitute for an interaction model, but it is a transparent heterogeneity
diagnostic. Segment-specific comparisons belong in the multiple-testing
plan if they affect decisions.

In [10]:
segment_analysis = GroupExperiment(
    executors=[
        GroupSizes(grouping_role=TreatmentRole()),
        OnRoleExperiment(
            executors=[
                GroupDifference(grouping_role=TreatmentRole()),
                TTest(grouping_role=TreatmentRole()),
            ],
            role=TargetRole(),
        ),
        ABAnalyzer(multitest_method=ABNTestMethodsEnum.holm),
    ],
    reporter=ABTestReporter(invert_pass=True),
    searching_role=GroupingRole(),
)
segmented = segment_analysis.execute(prepared)
segment_id = next(
    key for key in segmented.analysis_tables if key.startswith("GroupExperiment")
)
segmented.analysis_tables[segment_id]

,retail feature,retail group,retail control mean,retail test mean,retail difference,retail difference %,retail ci lower,retail ci upper,retail TTest pass,retail TTest p-value,saas feature,saas group,saas control mean,saas test mean,saas difference,saas difference %,saas ci lower,saas ci upper,saas TTest pass,saas TTest p-value
0,revenue,1,129.188010,138.190835,9.002826,6.968778,5.108138,12.897513,OK,0.000007,revenue,1,128.966833,140.627044,11.660211,9.041248,6.310010,17.010413,OK,0.000024
1,converted,1,0.265979,0.287474,0.021495,8.081435,-0.034796,0.077786,NOT OK,0.454383,converted,1,0.276786,0.356557,0.079772,28.820730,-0.004344,0.163887,NOT OK,0.063688


### GroupExperiment and effect heterogeneity

For segment $g$, the pipeline estimates
$\widehat\tau_g=\bar Y_{g,t}-\bar Y_{g,c}$. This is a descriptive
heterogeneity map. It does not by itself test whether
$\tau_{\text{retail}}=\tau_{\text{saas}}$; that question calls for an
interaction or another explicit contrast.

`GroupExperiment` starts each segment from a clean state. Derived fields
from unrelated segments cannot leak into its calculations. Every segment
report is then prefixed and combined into one wide table.

More segments mean more opportunities for false discoveries. Treat these
columns as exploratory unless their families and corrections were planned.

## Where to go from here

- Need a new cleaning step? Subclass `Transformer` and return a new `Dataset`.
- Need a moment-based distributed test? Subclass `StatsHypothesisTesting`
  and declare `REQUIRED_STATS`.
- Need repeated randomisation? Use `CycledExperiment`.
- Need a parameter grid? Use `ParamsExperiment`; `IfParamsExperiment` adds
  an early-stopping rule.
- Need a user-facing shell? Compose `ExperimentShell` and `Output` exactly
  as above.

Keep pipeline order explicit: splitters before comparators, comparators
before analyzers, and reporters after all results they consume.

**Next:** [Pandas ↔ Spark and distributed execution](05_Spark_Backend.ipynb).

## A debugging field guide

| Symptom | First thing to inspect |
|---|---|
| A target is absent from the report | Its role and whether a filter changed it |
| A comparator returns an empty table | Pipeline order and `tmp_roles` |
| Two targets overwrite one another | Executor `key` and id construction |
| A categorical numeric model fails | `DummyEncoder` and `additional_fields` |
| Matched rows join incorrectly | Index preservation |
| A custom statistic never reaches `Output` | Reporter registration and `get_ids` |
| Spark behaves differently from pandas | Backend implementation and approximation notes |

### Design rules worth keeping

1. Put statistical logic in an executor, presentation logic in a reporter,
   and orchestration in an experiment.
2. Search by role and executor class; reserve physical names for dataset
   construction and presentation.
3. Return a new `Dataset` from transformers and preserve indexes everywhere.
4. Make ids stable and minimal: they are data addresses, not debug strings.
5. Document the estimand beside every custom statistic.